# Preparation

In [ ]:
!pip install datasets==3.6.0 transformers evaluate sentencepiece accelerate scikit-multilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from skmultilearn.model_selection import iterative_train_test_split
from datasets import Dataset, DatasetDict
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os

# DO NOT include the brackets, just the string inside the quotes
os.environ['KAGGLE_USERNAME'] = "mysterioucz"
os.environ['KAGGLE_KEY'] = "61589d30f7cb3a67f84e1a3b69b7add9"

# Try the list again
!kaggle competitions list

ref                                                                              deadline             category                reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3    2026-04-15 23:59:00  Featured         2,207,152 Usd       2478           False  
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection         2026-02-27 23:59:00  Research           200,000 Usd       1400           False  
https://www.kaggle.com/competitions/stanford-rna-3d-folding-2                    2026-03-25 23:59:00  Featured           100,000 Usd       1104           False  
https://www.kaggle.com/competitions/med-gemma-impact-challenge                   2026-02-24 23:59:00  Featured           100,000 Usd        489           False  
https://www.kaggle.com/compe

Download Data set

In [ ]:
!kaggle competitions download -c 2110446-dsde-2025-2

  0% 0.00/40.8M [00:00<?, ?B/s]
100% 40.8M/40.8M [00:00<00:00, 1.37GB/s]


In [ ]:
!unzip -q 2110446-dsde-2025-2.zip

# EDA

In [ ]:
!ls

2110446-dsde-2025-2.zip  sample_submission.csv	train.csv
sample_data		 test.csv


In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df_test = pd.read_csv('test.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306419 entries, 0 to 306418
Data columns (total 14 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   id                                     306419 non-null  int64 
 1   comment                                304284 non-null  object
 2   สำนักงานตำรวจแห่งชาติ                  306419 non-null  int64 
 3   การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย      306419 non-null  int64 
 4   สภาเด็กและเยาวชนกรุงเทพมหานคร          306419 non-null  int64 
 5   กรมควบคุมมลพิษ                         306419 non-null  int64 
 6   กรมสรรพสามิต                           306419 non-null  int64 
 7   การไฟฟ้านครหลวง                        306419 non-null  int64 
 8   กรมทางหลวง                             306419 non-null  int64 
 9   สำนักงานประกันสุขภาพแห่งชาติ           306419 non-null  int64 
 10  การประปานครหลวง                        306419 non-null  int64 
 11  

In [ ]:
df.head()

,id,comment,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,ทำไมปล่อยให้จุดพลุกันสนั่นหวั่นไหว,0,0,0,0,0,0,0,0,0,0,0,0
1,1,แจ้งว่าการจุดพลุต้องขออนุญาต ทำไมจุดกันมากมายข...,0,0,0,0,0,0,0,0,0,0,0,0
2,2,คาดว่ามีการจุดพลุไม่ขอทางกรุงเทพให้ถูกต้อง ส่ง...,0,0,0,0,0,0,0,0,0,0,0,0
3,3,ไม่แน่ใจ กทม อนุญาตให้ร้านชอคโกแลตวิลจุพลุถึงก...,0,0,0,0,0,0,0,0,0,0,0,0
4,4,ไม่ทราบใครจัดงานปีใหม่ละแวกนี้ เปิดเสียงเพลงดั...,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(df.columns[1:])

Index(['comment', 'สำนักงานตำรวจแห่งชาติ', 'การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย',
       'สภาเด็กและเยาวชนกรุงเทพมหานคร', 'กรมควบคุมมลพิษ', 'กรมสรรพสามิต',
       'การไฟฟ้านครหลวง', 'กรมทางหลวง', 'สำนักงานประกันสุขภาพแห่งชาติ',
       'การประปานครหลวง', 'คณะกรรมการการพัฒนาเศรษฐกิจ',
       'กระทรวงการท่องเที่ยวและกีฬา', 'สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200'],
      dtype='object')


In [ ]:
df_test.head()

,id,comment
0,0,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...
1,1,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...
2,2,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย
3,3,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย
4,4,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...


# Preprocess


## Clean Data

In [ ]:
def clean_data(df):
  df = df = df.drop(columns=['id'],axis=1) # remove id
  df = df.dropna()
  df = df.drop_duplicates()
  return df


In [ ]:
df = clean_data(df)


In [ ]:
df_test['comment'] = df_test['comment'].fillna("")

In [ ]:
classes = df.columns[1:].values
class2id = {class_:id for id, class_ in enumerate(classes)}
id2class = {id:class_ for class_, id in class2id.items()}
print(classes)

['สำนักงานตำรวจแห่งชาติ' 'การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย'
 'สภาเด็กและเยาวชนกรุงเทพมหานคร' 'กรมควบคุมมลพิษ' 'กรมสรรพสามิต'
 'การไฟฟ้านครหลวง' 'กรมทางหลวง' 'สำนักงานประกันสุขภาพแห่งชาติ'
 'การประปานครหลวง' 'คณะกรรมการการพัฒนาเศรษฐกิจ'
 'กระทรวงการท่องเที่ยวและกีฬา' 'สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200']


## Train Test Split

In [ ]:
X = df[['comment']].values
y = df[classes].values

np.random.seed(42)
X_train_arr, y_train_arr, X_eval_arr, y_eval_arr = iterative_train_test_split(X, y, test_size=0.2)

# 3. Reconstruct DataFrames
df_train = pd.DataFrame(X_train_arr, columns=['comment'])
df_train[classes] = y_train_arr

df_eval = pd.DataFrame(X_eval_arr, columns=['comment'])
df_eval[classes] = y_eval_arr

# 4. Convert to Hugging Face Datasets
hf_train = Dataset.from_pandas(df_train)
hf_eval = Dataset.from_pandas(df_eval)
hf_test = Dataset.from_pandas(df_test)

# 5. Map to DatasetDict
raw_datasets = DatasetDict({
    'train': hf_train,
    'eval': hf_eval,
    'test': hf_test  # Ensure df_test was defined earlier
})

In [ ]:
from transformers import AutoTokenizer

model_path = 'clicknext/phayathaibert'

tokenizer = AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

In [ ]:
def optimized_preprocess(batches):
    # 1. Tokenize the entire batch of texts at once
    tokenized_inputs = tokenizer(
        batches['comment'],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # 2. Extract labels only if they exist (Train/Eval)
    # We check if the first class column is present in the batch
    if classes[0] in batches:
        batch_labels = []
        for i in range(len(batches['comment'])):
            labels = [float(batches[col][i]) for col in classes]
            batch_labels.append(labels)
        tokenized_inputs['labels'] = batch_labels

    return tokenized_inputs

# Process Train, Eval, and Test
tokenized_train = raw_datasets['train'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)

tokenized_eval = raw_datasets['eval'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['eval'].column_names
)

# Test set usually doesn't have labels, so the function handles that
tokenized_test = raw_datasets['test'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['test'].column_names
)


Map:   0%|          | 0/220898 [00:00<?, ? examples/s]

Map:   0%|          | 0/55225 [00:00<?, ? examples/s]

Map:   0%|          | 0/37406 [00:00<?, ? examples/s]

In [ ]:
from datasets import DatasetDict

tokenized_datasets = DatasetDict({
    'train': tokenized_train,
    'eval': tokenized_eval,
    'test': tokenized_test
})

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metric

In [ ]:
import evaluate
import numpy as np

# Combine metrics with averaging methods for multilabel
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

best_thresholds = []

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # 1. Convert logits to probabilities
    probs = sigmoid(predictions)

    # 2. Define or use external thresholds
    # If best_thresholds isn't found, default to 0.5 for all
    thresholds = getattr(compute_metrics, "thresholds", [0.5] * labels.shape[1])

    # 3. Apply threshold per column (class)
    # This creates a boolean mask and converts to int
    y_pred = np.zeros(probs.shape)
    for i in range(labels.shape[1]):
        y_pred[:, i] = (probs[:, i] > thresholds[i]).astype(int)

    # 4. Compute metrics
    # For multi-label, we usually use 'micro' or 'macro' averaging
    # instead of flattening everything
    # return clf_metrics.compute(
    #     predictions=y_pred.reshape(-1),
    #     references=labels.astype(int).reshape(-1)
    # )
    return {
        "f1_macro": f1_score(labels, y_pred, average='macro', zero_division=0),
        "precision_macro": precision_score(labels, y_pred, average='macro', zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average='macro', zero_division=0),
        "accuracy": accuracy_score(labels, y_pred)
    }

compute_metrics.thresholds = [0.5] * len(classes)


# Model

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=len(classes),
    id2label=id2class,
    label2id=class2id,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: clicknext/phayathaibert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training

In [ ]:
# Monitor via wandb
report_to = "wandb"

training_args = TrainingArguments(
    output_dir="my_multi_label_classify_model",
    learning_rate=1e-4,
    per_device_train_batch_size=512,
    per_device_eval_batch_size=512,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=report_to,
    # Enable Mixed Precision for faster training on GPU
    bf16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

In [ ]:
%%time
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['eval'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6, 'bos_token_id': 5}.


Epoch,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Accuracy
1,No log,0.042708,0.218970,0.359201,0.210110,0.838443
2,0.059973,0.039383,0.282530,0.326776,0.264345,0.847098
3,0.037958,0.038482,0.285708,0.334304,0.263259,0.850303


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

CPU times: user 6min 52s, sys: 20.6 s, total: 7min 13s
Wall time: 7min 13s


TrainOutput(global_step=1296, training_loss=0.04570760844666281, metrics={'train_runtime': 432.6347, 'train_samples_per_second': 1531.763, 'train_steps_per_second': 2.996, 'total_flos': 4.359444324793344e+16, 'train_loss': 0.04570760844666281, 'epoch': 3.0})

In [ ]:
# Convert logs to a Pandas DataFrame
df_logs = pd.DataFrame(trainer.state.log_history)

# 2. Filter for evaluation logs
# (Evaluation rows have 'eval_loss', whereas training rows have 'loss')
eval_stats = df_logs.dropna(subset=['eval_loss'])

# 3. Print the specific metrics
# Note: your 'f1_macro' becomes 'eval_f1_macro' in the logs
columns_to_show = ['epoch', 'step', 'eval_loss', 'eval_f1_macro', 'eval_accuracy']

# Use .get() or check if columns exist to avoid errors if a metric hasn't run yet
available_columns = [col for col in columns_to_show if col in eval_stats.columns]
print(eval_stats[available_columns])

   epoch  step  eval_loss  eval_f1_macro  eval_accuracy
0    1.0   432   0.042708       0.218970       0.838443
2    2.0   864   0.039383       0.282530       0.847098
4    3.0  1296   0.038482       0.285708       0.850303


# Find threshold for each class

In [ ]:
# 1. Get predictions (probabilities) from your trainer
predictions = trainer.predict(tokenized_datasets['eval'])
probs = torch.sigmoid(torch.Tensor(predictions.predictions)).numpy()
y_true = predictions.label_ids

In [ ]:

# 2. Find optimal threshold per class
for i in range(len(classes)):
    thresholds = np.linspace(0, 0.9, 50)
    f1_scores = [f1_score(y_true[:, i], (probs[:, i] > t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1_scores)]
    best_thresholds.append(np.maximum(best_t,0.1))
    print(f"Best Threshold for {classes[i]}: {best_t:.4f}")

Best Threshold for สำนักงานตำรวจแห่งชาติ: 0.4224
Best Threshold for การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย: 0.2755
Best Threshold for สภาเด็กและเยาวชนกรุงเทพมหานคร: 0.0000
Best Threshold for กรมควบคุมมลพิษ: 0.1653
Best Threshold for กรมสรรพสามิต: 0.0000
Best Threshold for การไฟฟ้านครหลวง: 0.3490
Best Threshold for กรมทางหลวง: 0.4041
Best Threshold for สำนักงานประกันสุขภาพแห่งชาติ: 0.0000
Best Threshold for การประปานครหลวง: 0.4224
Best Threshold for คณะกรรมการการพัฒนาเศรษฐกิจ: 0.0000
Best Threshold for กระทรวงการท่องเที่ยวและกีฬา: 0.0000
Best Threshold for สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200: 0.4776


In [ ]:
# Check how many actual samples you have per class in the eval set
support = y_true.sum(axis=0)
for i, class_name in enumerate(classes):
    print(f"{class_name}: {support[i]} samples")

สำนักงานตำรวจแห่งชาติ: 6436.0 samples
การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย: 510.0 samples
สภาเด็กและเยาวชนกรุงเทพมหานคร: 6.0 samples
กรมควบคุมมลพิษ: 383.0 samples
กรมสรรพสามิต: 9.0 samples
การไฟฟ้านครหลวง: 3757.0 samples
กรมทางหลวง: 1425.0 samples
สำนักงานประกันสุขภาพแห่งชาติ: 14.0 samples
การประปานครหลวง: 572.0 samples
คณะกรรมการการพัฒนาเศรษฐกิจ: 6.0 samples
กระทรวงการท่องเที่ยวและกีฬา: 8.0 samples
สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200: 865.0 samples


# Save model & Config

In [ ]:
trainer.save_model("my_multi_label_classify_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluation

In [ ]:
probs.shape

(55225, 12)

In [ ]:
import torch

# 1. Get predictions (logits) from the A100-powered trainer
preds_output = trainer.predict(tokenized_datasets['test'])

# 2. Convert logits to probabilities using Sigmoid
probs = torch.sigmoid(torch.tensor(preds_output.predictions)).numpy()

# 3. Create an empty matrix for binary predictions
y_pred = np.zeros(probs.shape, dtype=int)

# 4. Apply each class-specific threshold
# best_thresholds should be the list you generated earlier
for i, threshold in enumerate(best_thresholds):
    y_pred[:, i] = (probs[:, i] > threshold).astype(int)

In [ ]:
y_pred.shape

In [ ]:
submit_sample = pd.read_csv('sample_submission.csv')
submit_sample.head()

,id,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
prediction_df = pd.DataFrame(y_pred, columns=classes)

# Check if lengths match
if len(prediction_df) == len(submit_sample):
    # Assign predictions to the submission template
    submit_sample[classes] = prediction_df

    # Save to CSV
    submit_sample.to_csv('submission.csv', index=False)
    print("Submission saved to 'submission.csv'")
    display(submit_sample.head())

Submission saved to 'submission.csv'


,id,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,1,0,0,0,0,0


In [ ]:
from google.colab import files

# To download your final submission CSV
files.download('submission.csv')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: my_multi_label_classify_model/ (stored 0%)
  adding: my_multi_label_classify_model/config.json (deflated 79%)
  adding: my_multi_label_classify_model/checkpoint-1728/ (stored 0%)
  adding: my_multi_label_classify_model/checkpoint-1728/config.json (deflated 79%)
  adding: my_multi_label_classify_model/checkpoint-1728/rng_state.pth (deflated 26%)
  adding: my_multi_label_classify_model/checkpoint-1728/tokenizer_config.json (deflated 50%)
  adding: my_multi_label_classify_model/checkpoint-1728/optimizer.pt1
1
123



zip error: Interrupted (aborting)


FileNotFoundError: Cannot find file: my_model.zip

In [ ]:
# To download your model (it's best to zip it first since it's a folder)
!zip -r my_model.zip my_multi_label_classify_model/
files.download('my_model.zip')

  adding: my_multi_label_classify_model/ (stored 0%)
  adding: my_multi_label_classify_model/config.json (deflated 79%)
  adding: my_multi_label_classify_model/checkpoint-1728/ (stored 0%)
  adding: my_multi_label_classify_model/checkpoint-1728/config.json (deflated 79%)
  adding: my_multi_label_classify_model/checkpoint-1728/rng_state.pth (deflated 26%)
  adding: my_multi_label_classify_model/checkpoint-1728/tokenizer_config.json (deflated 50%)
  adding: my_multi_label_classify_model/checkpoint-1728/optimizer.pt (deflated 65%)
  adding: my_multi_label_classify_model/checkpoint-1728/scaler.pt (deflated 64%)
  adding: my_multi_label_classify_model/checkpoint-1728/training_args.bin (deflated 53%)
  adding: my_multi_label_classify_model/checkpoint-1728/tokenizer.json (deflated 77%)
  adding: my_multi_label_classify_model/checkpoint-1728/scheduler.pt (deflated 61%)
  adding: my_multi_label_classify_model/checkpoint-1728/trainer_state.json (deflated 67%)
  adding: my_multi_label_classify_mo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Accuracy

In [ ]:
# loss graph
train_loss = []
val_loss = []
val_acc = []
last_train_step = len(model.train_losses) // model.trainer.max_epochs
last_val_step = len(model.val_losses) // model.trainer.max_epochs

for epoch in range(model.trainer.max_epochs):
    start_train_idx = epoch * last_train_step
    end_train_idx = start_train_idx + last_train_step
    start_val_idx = epoch * last_val_step
    end_val_idx = start_val_idx + last_val_step
    train_loss_epoch = sum(model.train_losses[start_train_idx:end_train_idx]) / last_train_step
    val_loss_epoch = sum(model.val_losses[start_val_idx:end_val_idx]) / last_val_step
    val_acc_epoch = sum(model.val_accuaracy[start_val_idx:end_val_idx]) / last_val_step
    train_loss.append(train_loss_epoch)
    val_loss.append(val_loss_epoch)
    val_acc.append(val_acc_epoch)

plt.figure(figsize=(10, 5))
plt.plot(range(1, model.trainer.max_epochs + 1), train_loss, label='Training Loss')
plt.plot(range(1, model.trainer.max_epochs + 1), val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Per Epoch')
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(1, model.trainer.max_epochs + 1), val_acc, label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy Per Epoch')
plt.legend()
plt.show()